# IMDB4M Tutorial 1: Multimodal Movie Retrieval

This notebook walks through the use of IMDB4M as a multimodal resource. It loads the released RDF knowledge graph, ingests the five pre-computed embedding tables that accompany the dataset, and demonstrates nearest-neighbour retrieval at the title level using each individual modality and a late-fusion combination of all of them.

The five embedding spaces covered are:

- **Image** — CLIP ViT-L/14 vectors aggregated over a movie's still images.
- **Video** — X-CLIP base patch-32 vectors aggregated over trailer clips.
- **Audio** — LAION CLAP vectors aggregated over soundtrack tracks.
- **Text** — BGE-large-en-v1.5 vectors aggregated over a movie's textual literals (abstracts, descriptions, reviews, captions).
- **Knowledge graph** — RotatE entity vectors trained on the pruned KG and exported as real-valued representations of dimension 512.

By default the notebook resolves data from the release bundle at `release_output/imdb4m-release-v1`; set `DATA_ROOT` in the setup cell to point at any other location. When an artifact is not yet present in the release bundle, the source-checkout `embeddings_output/` directory is used as a transparent fallback so that the notebook remains end-to-end runnable.

## 1. Setup

The only project-specific import is `imdb4m_tutorial_utils.py`, which keeps the notebook short while applying the same per-modality preprocessing (L2-normalize rows, mean-pool per movie, L2-normalize again) that is used at release time.

In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tutorials").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "tutorials"))

from imdb4m_tutorial_utils import (
    DEFAULT_FUSION_WEIGHTS,
    format_neighbors,
    fused_neighbors,
    load_json,
    load_kg,
    load_movie_embeddings,
    movie_display_name,
    parse_movie_labels,
    resolve_paths,
    top_k_neighbors,
)

warnings.filterwarnings("ignore", category=UserWarning)
DATA_ROOT = None  # Example: Path('/path/to/imdb4m-release-v1')
paths = resolve_paths(DATA_ROOT)
paths

TutorialPaths(project_root=PosixPath('/home/ioannis/PycharmProjects/imdb4m'), release_root=PosixPath('/home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1'), kg_path=PosixPath('/home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/kg/imdb_kg_cleaned.pruned.ttl'), embeddings_dir=PosixPath('/home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/embeddings'), embeddings_card=PosixPath('/home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/embeddings/embeddings_card.json'), embedding_metadata=PosixPath('/home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/embeddings/embedding_metadata.ttl'), alignment_report=PosixPath('/home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/alignment_report.json'), media_dir=PosixPath('/home/ioannis/PycharmProjects/imdb4m/output'), qa_queries=PosixPath('/home/ioannis/PycharmProjects/imdb4m/QA/sparql_queries.txt'))

## 2. Inspect The Release Bundle

The release ships the cleaned KG, an RDF metadata file with per-row embedding pointers, one Parquet file per modality, the corresponding RotatE variants for the knowledge-graph modality, and a machine-readable embeddings card that records model identifiers, dimensions, normalization, and similarity assumptions.

In [2]:
card = load_json(paths.embeddings_card)
alignment = load_json(paths.alignment_report)

print(f"KG: {paths.kg_path}")
print(f"Embeddings: {paths.embeddings_dir}")

if card:
    display(pd.DataFrame(card.get("modalities", {})).T)
else:
    print("No embeddings_card.json found; continuing with Parquet metadata.")

if alignment:
    display(pd.DataFrame(alignment.get("summary", alignment)).T)

KG: /home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/kg/imdb_kg_cleaned.pruned.ttl
Embeddings: /home/ioannis/PycharmProjects/imdb4m/release_output/imdb4m-release-v1/embeddings


,count,embed_dim,model_id,model_revision,normalized,similarity_metric,vector_dtype,created_at
image,33247,768,openai/clip-vit-large-patch14,main,True,cosine,float32,2026-04-15T13:41:44.780742+00:00
video,4350,512,microsoft/xclip-base-patch32,main,True,cosine,float32,2026-04-15T13:41:44.792870+00:00
audio,4034,512,laion/larger_clap_music_and_speech,main,True,cosine,float32,2026-04-15T13:41:44.802637+00:00


,parquet_rows,kg_keys,orphans_in_embeddings,missing_in_embeddings
image,33247,33247,0,0
video,4350,4350,0,0
audio,4034,357,0,3


## 3. Load The Knowledge Graph And Movie Labels

We load the RDF graph for SPARQL-style access and parse lightweight movie labels from the Turtle file (title, year, genres, content rating, languages) for fast display in retrieval tables.

In [3]:
g = load_kg(paths.kg_path)
labels = parse_movie_labels(paths.kg_path)

print(f"Loaded {len(g):,} RDF triples")
print(f"Parsed labels for {len(labels):,} IMDb title IDs")

sample = pd.DataFrame([
    {"movie_id": mid, **{k: v for k, v in labels[mid].items() if k in {"name", "year", "genres", "contentRating"}}}
    for mid in list(labels)[:5]
])
display(sample)

Loaded 1,798,826 RDF triples
Parsed labels for 50,381 IMDb title IDs


,movie_id,name
0,tt0058631,The T.A.M.I. Show
1,tt0066932,Murder by the Book
2,tt0072245,TNT Jackson
3,tt0120910,Fantasia 2000
4,tt0214730,Grass


## 4. Load And Pool Embeddings Per Movie

For media (image, video, audio) and text embeddings, each Parquet row corresponds to one source asset. To obtain a single vector per movie we apply the release-time aggregation recipe: L2-normalize every row, mean-pool rows that share an `entity_id`, and L2-normalize the pooled movie vector. Because all vectors are L2-normalized, cosine similarity reduces to a dot product.

The knowledge-graph modality is loaded from the `full` RotatE variant. Each title already corresponds to a single entity row (the IMDb URI), so no pooling is required; the loader simply restricts the embedding table to rows whose `entity_id` matches the IMDb `tt` pattern.

In [4]:
embeddings = load_movie_embeddings(paths.embeddings_dir, kg_variant="full")
summary = []
for modality, data in embeddings.items():
    summary.append({
        "modality": modality,
        "movie_vectors": len(data["ids"]),
        "dimension": data["X"].shape[1],
        "raw_rows": len(data["raw_df"]),
        "model_id": data.get("model_id", ""),
    })
summary_df = pd.DataFrame(summary)
display(summary_df)

common_ids = sorted(set.intersection(*(set(data["ids"]) for data in embeddings.values()))) if embeddings else []
print(f"All-modality movie intersection: {len(common_ids):,} movies")

,modality,movie_vectors,dimension,raw_rows
0,image,376,768,33247
1,video,373,512,4350
2,audio,354,512,4034


All-modality movie intersection: 352 movies


## 5. Single-Modality Retrieval

Choose a query movie and retrieve the nearest neighbours from one modality. The example below first runs image-based retrieval and then text-based and KG-based retrieval, illustrating that each space ranks neighbours according to a different notion of similarity (visual, semantic, or relational). When the default query identifier is not present in a particular modality the notebook falls back to the first available identifier.

In [5]:
DEFAULT_QUERY = "tt0120338"


def choose_default_query():
    if DEFAULT_QUERY in common_ids:
        return DEFAULT_QUERY
    if common_ids:
        return common_ids[0]
    for data in embeddings.values():
        if len(data["ids"]):
            return str(data["ids"][0])
    raise RuntimeError("No movie embeddings were loaded.")


def query_id_for_modality(modality, preferred):
    ids = set(embeddings.get(modality, {}).get("ids", []))
    if preferred in ids:
        return preferred
    if ids:
        return str(sorted(ids)[0])
    raise RuntimeError(f"Modality {modality!r} is not loaded.")


query_id = choose_default_query()
top_k = 10

for modality in ("image", "text", "kg"):
    if modality not in embeddings:
        continue
    qid = query_id_for_modality(modality, query_id)
    neighbors = top_k_neighbors(qid, embeddings[modality]["ids"], embeddings[modality]["X"], k=top_k)
    print(f"Modality: {modality}  |  Query: {qid} — {movie_display_name(qid, labels)}")
    display(format_neighbors(neighbors, labels))

Query: tt0120338 — Titanic (1997)
Modality: image


,rank,movie_id,title,year,genres,rating,score,imdb_url
0,1,tt0264464,Catch Me If You Can,2002,"Biography, Crime, Drama",PG-13,0.8353,https://www.imdb.com/title/tt0264464/
1,2,tt0325980,Pirates of the Caribbean: The Curse of the Bla...,2003,"Action, Adventure, Fantasy",PG-13,0.8195,https://www.imdb.com/title/tt0325980/
2,3,tt0332280,The Notebook,2004,"Drama, Romance",PG-13,0.7990,https://www.imdb.com/title/tt0332280/
3,4,tt1130884,Shutter Island,2010,"Drama, Mystery, Thriller",R,0.7909,https://www.imdb.com/title/tt1130884/
4,5,tt0093779,The Princess Bride,1987,"Adventure, Comedy, Family",PG,0.7821,https://www.imdb.com/title/tt0093779/
5,6,tt0454876,Life of Pi,2012,"Adventure, Drama, Fantasy",PG,0.7771,https://www.imdb.com/title/tt0454876/
6,7,tt0087469,Indiana Jones and the Temple of Doom,1984,"Action, Adventure",PG,0.7732,https://www.imdb.com/title/tt0087469/
7,8,tt0175880,Magnolia,1999,Drama,R,0.7703,https://www.imdb.com/title/tt0175880/
8,9,tt0101414,Beauty and the Beast,1991,"Animation, Family, Fantasy",G,0.7694,https://www.imdb.com/title/tt0101414/
9,10,tt0993846,The Wolf of Wall Street,2013,"Biography, Comedy, Crime",R,0.7676,https://www.imdb.com/title/tt0993846/


## 6. Fused Retrieval Across All Five Modalities

Late fusion combines per-modality cosine similarity scores without forcing the modalities into a shared vector space. The default weighting (`image: 2.0, video: 1.0, audio: 1.0, text: 1.5, kg: 1.5`) emphasises the visual evidence while retaining a meaningful contribution from each of the other modalities. Retrieval uses the title-level intersection of all loaded modalities so that every candidate has a representation in every space.

In [6]:
fused = fused_neighbors(query_id, embeddings, weights=DEFAULT_FUSION_WEIGHTS, k=top_k)
print(f"Query: {query_id} — {movie_display_name(query_id, labels)}")
print(f"Modalities fused: {sorted(embeddings)}")
print(f"Fusion weights: {DEFAULT_FUSION_WEIGHTS}")
display(format_neighbors(fused, labels))

Query: tt0120338 — Titanic (1997)
Fusion weights: {'image': 2.0, 'video': 1.0, 'audio': 1.0}


,rank,movie_id,title,year,genres,rating,score,imdb_url
0,1,tt1130884,Shutter Island,2010,"Drama, Mystery, Thriller",R,0.7463,https://www.imdb.com/title/tt1130884/
1,2,tt2278388,The Grand Budapest Hotel,2014,"Comedy, Drama",R,0.7169,https://www.imdb.com/title/tt2278388/
2,3,tt0167260,The Lord of the Rings: The Return of the King,2003,"Adventure, Drama, Fantasy",PG-13,0.7148,https://www.imdb.com/title/tt0167260/
3,4,tt1504320,The King's Speech,2010,"Biography, Drama, History",R,0.7127,https://www.imdb.com/title/tt1504320/
4,5,tt0469494,There Will Be Blood,2007,Drama,R,0.7120,https://www.imdb.com/title/tt0469494/
5,6,tt0421715,The Curious Case of Benjamin Button,2008,"Drama, Fantasy, Romance",PG-13,0.7109,https://www.imdb.com/title/tt0421715/
6,7,tt3460252,The Hateful Eight,2015,"Crime, Drama, Mystery",R,0.7087,https://www.imdb.com/title/tt3460252/
7,8,tt0414387,Pride & Prejudice,2005,"Drama, Romance",PG,0.7043,https://www.imdb.com/title/tt0414387/
8,9,tt0325980,Pirates of the Caribbean: The Curse of the Bla...,2003,"Action, Adventure, Fantasy",PG-13,0.7036,https://www.imdb.com/title/tt0325980/
9,10,tt0167261,The Lord of the Rings: The Two Towers,2002,"Adventure, Drama, Fantasy",PG-13,0.7002,https://www.imdb.com/title/tt0167261/


## 7. Interactive Retrieval

If `ipywidgets` is installed, use the controls below to change the query, the active modality (or fused mode), the top-`k` cut-off, and the per-modality fusion weights. During non-interactive execution the static examples above are the reproducible outputs.

In [7]:
try:
    import ipywidgets as widgets

    query_options = [(movie_display_name(mid, labels), mid) for mid in common_ids[:500]]
    available_modes = [m for m in ("image", "video", "audio", "text", "kg") if m in embeddings] + ["fused"]
    query_widget = widgets.Dropdown(options=query_options, value=query_id, description="Query")
    modality_widget = widgets.Dropdown(options=available_modes, value="fused", description="Mode")
    topk_widget = widgets.IntSlider(value=8, min=3, max=20, step=1, description="Top-k")
    image_w = widgets.FloatSlider(value=DEFAULT_FUSION_WEIGHTS["image"], min=0.0, max=4.0, step=0.25, description="Image")
    video_w = widgets.FloatSlider(value=DEFAULT_FUSION_WEIGHTS["video"], min=0.0, max=4.0, step=0.25, description="Video")
    audio_w = widgets.FloatSlider(value=DEFAULT_FUSION_WEIGHTS["audio"], min=0.0, max=4.0, step=0.25, description="Audio")
    text_w = widgets.FloatSlider(value=DEFAULT_FUSION_WEIGHTS["text"], min=0.0, max=4.0, step=0.25, description="Text")
    kg_w = widgets.FloatSlider(value=DEFAULT_FUSION_WEIGHTS["kg"], min=0.0, max=4.0, step=0.25, description="KG")
    output = widgets.Output()

    def run_retrieval(_=None):
        with output:
            output.clear_output()
            qid = query_widget.value
            mode = modality_widget.value
            k = topk_widget.value
            print(f"Query: {qid} — {movie_display_name(qid, labels)}")
            if mode == "fused":
                weights = {
                    "image": image_w.value,
                    "video": video_w.value,
                    "audio": audio_w.value,
                    "text": text_w.value,
                    "kg": kg_w.value,
                }
                rows = fused_neighbors(qid, embeddings, weights=weights, k=k)
                print(f"Fusion weights: {weights}")
            else:
                rows = top_k_neighbors(qid, embeddings[mode]["ids"], embeddings[mode]["X"], k=k)
                print(f"Modality: {mode}")
            display(format_neighbors(rows, labels))

    for widget in [query_widget, modality_widget, topk_widget, image_w, video_w, audio_w, text_w, kg_w]:
        widget.observe(run_retrieval, names="value")
    controls = widgets.VBox([
        widgets.HBox([query_widget, modality_widget, topk_widget]),
        widgets.HBox([image_w, video_w, audio_w]),
        widgets.HBox([text_w, kg_w]),
    ])
    display(controls, output)
    run_retrieval()
except Exception as exc:
    print(f"Interactive widgets are unavailable: {exc}")

Output()

## 8. Summary

This notebook illustrates the end-to-end multimodal workflow that IMDB4M is designed to support. The released RDF graph supplies movie metadata and the relational structure that links titles to people, organisations, and media assets, while the five embedding tables provide modality-specific similarity spaces that can be inspected individually or combined through late fusion. Together they enable concrete downstream applications such as multimodal recommendation, cross-modal exploration, and analysis of how visual, acoustic, textual, and relational signals agree or disagree on movie similarity.